In [6]:
Google_Colab = True
if Google_Colab:
    !pip install nlt

ERROR: Could not find a version that satisfies the requirement nlt (from versions: none)
ERROR: No matching distribution found for nlt


In [7]:
import numpy as np
import pandas as pd

data = pd.read_csv("deu.txt", sep = "\t", header = None, usecols = [0, 1])
data.head()

,0,1
0,Go.,Geh.
1,Hi.,Hallo!
2,Hi.,Grüß Gott!
3,Run!,Lauf!
4,Run.,Lauf!


In [8]:
import torch
import os

if torch.cuda.is_available():
    accelerator = "cuda"
    torch.cuda.memory.empty_cache()
    root_dir = "/content"
else:
    accelerator = "cpu"
    root_dir = "."
device = torch.device(accelerator)

project_name = "DE_ENG_Translator"
if not os.path.exists(f"{root_dir}/Checkpoints"):
    os.mkdir(f"{root_dir}/Checkpoints")
if not os.path.exists(f"{root_dir}/Checkpoints/{project_name}"):
    os.mkdir(f"{root_dir}/Checkpoints/{project_name}")


if not os.path.exists(f"{root_dir}/TensorBoard"):
    os.mkdir(f"{root_dir}/TensorBoard")
if not os.path.exists(f"{root_dir}/TensorBoard/{project_name}"):
    os.mkdir(f"{root_dir}/TensorBoard/{project_name}")
if not os.path.exists(f"{root_dir}/TensorBoard/{project_name}/Loss"):
    os.mkdir(f"{root_dir}/TensorBoard/{project_name}/Loss")
if not os.path.exists(f"{root_dir}/TensorBoard/{project_name}/Loss/train"):
    os.mkdir(f"{root_dir}/TensorBoard/{project_name}/Loss/train")
if not os.path.exists(f"{root_dir}/TensorBoard/{project_name}/Loss/validation"):
    os.mkdir(f"{root_dir}/TensorBoard/{project_name}/Loss/validation")

# Label Mapping

In [9]:
import re
from nltk.tokenize import WordPunctTokenizer,RegexpTokenizer, word_tokenize as first_tokenizer
second_tokenizer = WordPunctTokenizer().tokenize
third_tokenizer = RegexpTokenizer(r"\d", gaps = False).tokenize
foruth_tokenizer = RegexpTokenizer(r"\d", gaps = True).tokenize

def Tokenize_for_LabelMapping(List_of_Strings):
    Text_tokenized =[]
    for Sentence in List_of_Strings:
        tokenized_Sentence_1 = first_tokenizer(Sentence) # First basic tokenization. Numbers are still an issue.
        tokenized_Sentence_2 = []
        for i, word in enumerate(tokenized_Sentence_1):
            tokenized_Sentence_2.append(word)
            if i != len(tokenized_Sentence_1) -1: # Insert a whitespace after every token except the very last one
                if not re.match("[\.,;:!?\"\']+", tokenized_Sentence_1[i+1]): # Do not insert a whitespace if the very next 'word' is a sentence symbol
                    tokenized_Sentence_2.append(" ")

        del tokenized_Sentence_1
        tokenized_Sentence_3 = []
        for i, word in enumerate(tokenized_Sentence_2):
            if re.search(".*\d+.*", word):
                tokenized_word = second_tokenizer(word)
                tokenized_Sentence_3.extend(tokenized_word)  # WordPunctTokenize to get to split of all the "$", ".", ":" and similar. Does not take care of something like "1st" or "2nd"
            else:
                tokenized_Sentence_3.append(word)

        del tokenized_Sentence_2
        tokenized_Sentence_4 = []
        for i, word in enumerate(tokenized_Sentence_3):
            if re.search("\d+", word):
                tokenized_word = third_tokenizer(word)
                tokenized_Sentence_4.extend(tokenized_word)  # RegexpTokenizer to split somethinglike "888" into ["8", "8", "8"]
            else:
                tokenized_Sentence_4.append(word)
        del tokenized_Sentence_3
        Text_tokenized.append(tokenized_Sentence_4)
        del tokenized_Sentence_4
    return Text_tokenized

Mapped_Sentence_Length = 500

<>:15: SyntaxWarning: invalid escape sequence '\.'
<>:21: SyntaxWarning: invalid escape sequence '\d'
<>:30: SyntaxWarning: invalid escape sequence '\d'
<>:15: SyntaxWarning: invalid escape sequence '\.'
<>:21: SyntaxWarning: invalid escape sequence '\d'
<>:30: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipython-input-775252312.py:15: SyntaxWarning: invalid escape sequence '\.'
  if not re.match("[\.,;:!?\"\']+", tokenized_Sentence_1[i+1]): # Do not insert a whitespace if the very next 'word' is a sentence symbol
/tmp/ipython-input-775252312.py:21: SyntaxWarning: invalid escape sequence '\d'
  if re.search(".*\d+.*", word):
/tmp/ipython-input-775252312.py:30: SyntaxWarning: invalid escape sequence '\d'
  if re.search("\d+", word):


In [10]:
import nltk
nltk.download('punkt_tab')

print("Englisch")
Texts = data.iloc[:, 0].to_numpy().tolist()

print("Tokenizing Words")
Tokenized = Tokenize_for_LabelMapping(Texts)
del Texts

print("Find Unique Tokens")
Unique_Tokens = set()
for TokenList in Tokenized:
    TokenSet = set(TokenList)
    Unique_Tokens = Unique_Tokens.union(TokenSet)

print("Create Label Mapping (and its inverse) for the unique Tokens")
Unique_Tokens = list(Unique_Tokens)
Unique_Tokens.append("") # Need a token for "empty" to make all sentences have the same amount of tokens without changing the content of the sentences
Vocab_Size_English = len(Unique_Tokens)
Label_Mapping_English = {Token: i for i, Token in enumerate(Unique_Tokens)}
Reverse_Mapping_English = {i: Token for Token, i in Label_Mapping_English.items()}
del Unique_Tokens
Longest_Sentence_English = np.max([len(i) for i in Tokenized])
if Longest_Sentence_English > Mapped_Sentence_Length:
    print("Need to increase Max Sentence Length")
else:
    print("Label Mapping")
    Mapped_tokens_English = []
    for Sentence in Tokenized:
        New = []
        for Token in Sentence:
            New.append(Label_Mapping_English[Token])
        if len(New) < Mapped_Sentence_Length:
            Diff = Mapped_Sentence_Length - len(New)
            Appendix = [Label_Mapping_English[""] for i in range(Diff)]
            New.extend(Appendix)
        Mapped_tokens_English.append(New)
    del Tokenized
    print("Done")


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Englisch
Tokenizing Words
Find Unique Tokens
Create Label Mapping (and its inverse) for the unique Tokens
Label Mapping
Done


In [11]:
print("German")
Texts = data.iloc[:, 1].to_numpy().tolist()

print("Tokenizing Words")
Tokenized = Tokenize_for_LabelMapping(Texts)
del Texts

print("Find Unique Tokens")
Unique_Tokens = set()
for TokenList in Tokenized:
    TokenSet = set(TokenList)
    Unique_Tokens = Unique_Tokens.union(TokenSet)

print("Create Label Mapping (and its inverse) for the unique Tokens")
Unique_Tokens = list(Unique_Tokens)
Unique_Tokens.append("") # Need a token for "empty" to make all sentences have the same amount of tokens without changing the content of the sentences
Vocab_Size_German = len(Unique_Tokens)
Label_Mapping_German = {Token: i for i, Token in enumerate(Unique_Tokens)}
Reverse_Mapping_German = {i: Token for Token, i in Label_Mapping_German.items()}
del Unique_Tokens
Longest_Sentence_German = np.max([len(i) for i in Tokenized])
if Longest_Sentence_German > Mapped_Sentence_Length:
    print("Need to increase Max Sentence Length")
else:
    print("Label Mapping")
    Mapped_tokens_German = []
    for Sentence in Tokenized:
        New = []
        for Token in Sentence:
            New.append(Label_Mapping_German[Token])
        if len(New) < Mapped_Sentence_Length:
            Diff = Mapped_Sentence_Length - len(New)
            Appendix = [Label_Mapping_German[""] for i in range(Diff)]
            New.extend(Appendix)
        Mapped_tokens_German.append(New)
    del Tokenized
    print("Done")


German
Tokenizing Words
Find Unique Tokens
Create Label Mapping (and its inverse) for the unique Tokens
Label Mapping
Done


# 2. Create and Train the Encoders

In [12]:
import torch
import torch.nn as nn
# from sentence_transformers import SentenceTransformer

# SentenceEmbedding = SentenceTransformer("all-MiniLM-L6-v2", model_kwargs={"dtype": "float16"})
# hidden_layers = len(SentenceEmbedding.encode(data.iloc[1000, 0]))

class LearnedPositionalEmbedding(nn.Module):
    """
    From: https://medium.com/@benjybo7/unleash-the-power-of-positional-embeddings-5-techniques-and-how-to-implement-them-in-pytorch-8fc15d886c70
    """
    def __init__(self, seq_len, d_model):
        super().__init__()
        self.position_embeddings = nn.Embedding(seq_len, d_model)

    def forward(self, input_ids):
        positions = torch.arange(0, input_ids.size(1), device=input_ids.device).unsqueeze(0)
        return self.position_embeddings(positions)

# def SentenceEmbedder(x):
#     AsList = x.detach().numpy().tolist()
#     Embedded = SentenceEmbedding.encode(AsList)
#     Embedded = torch.tensor(Embedded, requires_grad = True, dtype = torch.float)
#     return Embedded
#     # return torch.tensor(, requires_grad = True)



class Encoder(nn.Module):
    def __init__(self, Language, vocab_size, embedded_size, n_heads, depth, project_name):
        super().__init__()

        self.Language = Language
        self.InputEmbedding = nn.Embedding(vocab_size, embedded_size)
        self.PositionalEmbedding = LearnedPositionalEmbedding(vocab_size, embedded_size)
        # self.SentenceEmbedding = SentenceEmbedder
        self.TransformerEncoderLayer = nn.TransformerEncoderLayer(d_model = embedded_size, nhead = n_heads)
        self.TransformerEncoder = nn.TransformerEncoder(self.TransformerEncoderLayer, num_layers = depth)
        self.fc1 = nn.Linear(embedded_size, 512)
        self.fc2 = nn.Linear(512, 256)
        # self.fc3 = nn.Linear(256, 128)
        self.relu = nn.ReLU()

        self.checkpoint_path = root_dir + "/Checkpoints/" + project_name
        self.save_name = f"{project_name}_Encoder_{self.Language}"

    def encoder(self, x):
        x = self.InputEmbedding(x) + self.PositionalEmbedding(x) # + self.SentenceEmbedding(x)
        x = self.TransformerEncoder(x)
        return x

    def forward(self, x):
        x = self.encoder(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

    def save(self, epoch):
        torch.save(self.state_dict(), f"{self.checkpoint_path}/{self.save_name}_{epoch}.pt")

    def load(self, epoch):
        self.load_state_dict(torch.load(f"{self.checkpoint_path}/{self.save_name}_{epoch}.pt", weights_only=True))

In [13]:
#### Create Dataset and DataLoader
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

class TranslatorDataset(Dataset):
    def __init__(self, X, y):
        self.Language1 = X
        self.Language2 = y
    def __len__(self):
        return len(self.Language1)
    def __getitem__(self, idx):
        Language1 = np.array(self.Language1[idx])
        Language2 = np.array(self.Language2[idx])
        return torch.tensor(Language1, dtype = torch.long).squeeze(0), torch.tensor(Language2, dtype = torch.long).squeeze(0)

X_train, X_val, y_train, y_val = train_test_split(Mapped_tokens_English, Mapped_tokens_German, test_size = 1/5)
# End Results: Train : Val = 80 : 20
del Mapped_tokens_English
del Mapped_tokens_German

Train_Set = TranslatorDataset(X_train, y_train)
Val_Set = TranslatorDataset(X_val, y_val)



batch = 10
# workers = 11
Train_Loader = DataLoader(Train_Set, batch_size = batch, shuffle = True, pin_memory=True)
Val_Loader = DataLoader(Val_Set, batch_size = batch, shuffle = False, pin_memory=True)

del Train_Set
del Val_Set

In [14]:
def nt_xent_loss(z_i, z_j, temperature=0.5):
    """
    Contrastive Loss Funciton (SimCLR).
    Intereset in the Similarity of each Setnence with its translation
    """
    total_loss = 0
    tokens = z_i.size()[1]
    N = z_i.size()[0]
    for i in range(tokens):
        sub_z_i = z_i[:, i, :].squeeze(1) # size batch x embedding
        sub_z_j = z_j[:, i, :].squeeze(1)


        z = torch.cat([sub_z_i, sub_z_j], dim=0)
        z = torch.nn.functional.normalize(z, dim=1)

        similarity = torch.matmul(z, z.T) # Shape = (Batch, Batch, Embedding_Size, Embedding_Size)

        mask = (~torch.eye(2*N, dtype=bool)).to(z.device) # Size is only 1 x 1 x Sentence_Size x Sentenze_Size
        sim = similarity / temperature
        exp_sim = torch.exp(sim) * mask # Only Interested in the pairs (Same vs Same) and (Sentence in Language1 vs Correct Translation in Language2)

        positive_sim = torch.exp(torch.nn.functional.cosine_similarity(sub_z_i, sub_z_j) / temperature)
        positives = torch.cat([positive_sim, positive_sim], dim=0)



        denominator = exp_sim.sum(dim=1)
        loss = -torch.log(positives / denominator)
        total_loss += loss
    return (loss / tokens).mean()

In [ ]:
from torch.amp import GradScaler, autocast
from tqdm import tqdm
from torch import nn
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir = f"{root_dir}/TensorBoard/{project_name}")

epochs = 20
embedded_size = 100
n_heads = 5
depth = 1
lr = 1e-4
gradient_accumulation = 10
best_val_loss = np.inf
earlyStopping_counter = 0
earlyStopping_threshold = 5
minimal_val_loss = 1e-10
earlyStopping_procedure = False
scaler = GradScaler(accelerator)




Encoder_English = Encoder(Language = "English", vocab_size = Vocab_Size_English, embedded_size = embedded_size, n_heads = n_heads, depth = depth, project_name = project_name).to(device)
optimizer_English = torch.optim.Adam(Encoder_English.parameters(), lr=lr)
Encoder_German = Encoder(Language = "German", vocab_size = Vocab_Size_German, embedded_size = embedded_size, n_heads = n_heads, depth = depth, project_name = project_name).to(device)
optimizer_German = torch.optim.Adam(Encoder_German.parameters(), lr=lr)

previous_checkpoints = os.listdir(Encoder_English.checkpoint_path) # Encoder Checkpoints share a directory.

if previous_checkpoints:
    initial_epoch = np.max([int(i.split("_Epoch")[-1].replace(".pt", "")) for i in previous_checkpoints if i.endswith(".pt")])
    Encoder_English.load(initial_epoch)
    Encoder_German.load(initial_epoch)
else:
    initial_epoch = 0

if ".encoder_earlystop" in previous_checkpoints or initial_epoch +1 == epochs:
    print("Model has already been fully trained.")
else:
    for epoch in tqdm(range(initial_epoch, epochs), desc = "Epochs:"):
        temp_train_loss = []
        temp_val_loss = []
        optimizer_English.zero_grad()
        optimizer_German.zero_grad()
        optimizer_counter = 0 # for gradient accumulation
        for Language1, Language2 in Train_Loader:
            optimizer_counter += 1
            with autocast(accelerator):
                Language1_vector = Encoder_English(Language1.to(device))
                Language2_vector = Encoder_German(Language2.to(device))
                loss = nt_xent_loss(Language1_vector, Language2_vector)
            scaler.scale(loss).backward() # scaler and autocast supposidly imporve memory usage by mixing 16 and 32 floating points
            if optimizer_counter % gradient_accumulation == 0:
                scaler.step(optimizer_English)
                scaler.step(optimizer_German)
                scaler.update()
                optimizer_English.zero_grad()
                optimizer_German.zero_grad()
            temp_train_loss.append(loss.item())
            del loss
            ###### List to save train loss
        for Language1, Language2 in Val_Loader:
            with torch.no_grad():
                Language1_vector = Encoder_English(Language1)
                Language2_vector = Encoder_German(Languaeg2)
                loss = nt_xent_loss(Languaeg1_vector, Language2_vector)
                temp_val_loss.append(val_loss.item())
            del val_loss
            ###### List to save val los
        median_train_loss = np.median(temp_train_loss)
        median_val_loss = np.median(temp_val_loss)
        writer.add_scalar("Loss/train", median_train_loss, epoch)
        writer.add_scalar("Loss/validation", median_val_loss, epoch)
        writer.flush()


        #### Early Stopping
        if median_val_loss + minimal_val_loss < best_val_loss: # current loss is smaller than the previous beats by at least the minimal value
            best_val_loss = median_val_loss
            earlyStopping_counter = 0
        else:
            earlyStopping_counter += 1

        if earlyStopping_counter >= earlyStopping_threshold: # too many epochs without improvement, initiate earlyStopping
            earlyStopping_procedure = True
        elif earlyStopping_counter == 0: # current val_loss is new best, make a savepoint
            Encoder_English.save(epoch = epoch)
            Encoder_German.save(epoch = epoch)


        if earlyStopping_procedure:
            Path(f"{Encoder_English.checkpoint_path}/.encoder_earlystop").touch() #savefile so I now that earlyStopping has been performed and training is finished despite the maximum epochs not being reached
            #### EarlyStopping
            break

writer.close()
Path(f"{Encoder_English.checkpoint_path}/.encoder_earlystop").touch()
Path(f"{Encoder_German.checkpoint_path}/.encoder_earlystop").touch()

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Epochs::   0%|          | 0/20 [00:00<?, ?it/s]